# Chapter 2: Model Architectures

This notebook explores transformer architecture, LLM families, and model selection.

## Transformer Architecture Fundamentals

In [ ]:
import torch
import torch.nn as nn

# Simplified self-attention mechanism
class SelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)
        
        # Reshape for multi-head
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = torch.softmax(scores, dim=-1)
        
        # Apply attention to values
        output = torch.matmul(attn_weights, V)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        
        return self.out_proj(output)

# Test
attn = SelfAttention(embed_dim=512, num_heads=8)
x = torch.randn(1, 10, 512)  # batch_size=1, seq_len=10, embed_dim=512
output = attn(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

## LLM Families Comparison

In [ ]:
# Compare LLM families
llm_families = [
    {
        "name": "LLaMA 2",
        "sizes": ["7B", "13B", "70B"],
        "license": "Research",
        "strengths": ["Strong performance", "Open weights", "Good documentation"]
    },
    {
        "name": "Mistral 7B",
        "sizes": ["7B"],
        "license": "Apache 2.0",
        "strengths": ["Efficient", "Strong performance", "Permissive license"]
    },
    {
        "name": "GPT-4",
        "sizes": ["Unknown"],
        "license": "Proprietary",
        "strengths": ["SOTA performance", "Strong reasoning", "API access"]
    }
]

for family in llm_families:
    print(f"\n{family['name']}")
    print(f"  Sizes: {', '.join(family['sizes'])}")
    print(f"  License: {family['license']}")
    print(f"  Strengths: {', '.join(family['strengths'])}")

## Parameter Count Estimation

In [ ]:
def estimate_parameters(num_layers, d_model, d_ff, vocab_size):
    """Estimate parameter count for a transformer model"""
    # Embedding parameters
    embedding_params = vocab_size * d_model
    
    # Transformer layer parameters
    # Attention: Q, K, V, O projections
    attention_params = 4 * d_model * d_model
    
    # Feed-forward: up, down, gate projections
    ffn_params = 3 * d_model * d_ff
    
    # Layer norms
    layer_norm_params = 4 * 2 * d_model  # 2 per layer (attention + FFN)
    
    # Total per layer
    layer_params = attention_params + ffn_params + layer_norm_params
    
    # Total model
    total_params = embedding_params + num_layers * layer_params
    
    return total_params

# Example: 7B model
params = estimate_parameters(
    num_layers=32,
    d_model=4096,
    d_ff=16384,
    vocab_size=32000
)

print(f"Estimated parameters: {params / 1e9:.2f}B")

## Key Takeaways

1. Transformer architecture is the foundation of LLMs
2. Self-attention enables context understanding
3. Different LLM families have different strengths
4. Model size impacts performance and resource requirements
5. Efficient architectures (MoE, sliding window) improve performance